Notebook para generar embeddings
Previamente ya se realizaron para pocos archivos usando un modelos de OpenAI y validando con Qdrant [rag_qdrant](https://github.com/Halsey26/embedding_PerAI/blob/main/rag_qdrant.ipynb)
Sin embargo, ahora son más de 30 archivos pdf, algunos incluso con 300 páginas. Por ende se plantea usar langchain para:
- Chunkenizado
- Embedding
- Almacenamiento - Qdrant
- Función búsqueda
Después se modularizará para detectar los pdfs y obtener los embeddings

Librerias para descargar
- %pip install -qU pypdf
- pip install langchain
- pip install langchain-community
- pip install sentence-transformers
- pip intall tiktoken

## fsdf
Detecta si un pdf ya ha sido procesado. Si en caso no ha sido procesado, se aplica las funciones y se marca como **hecho**.

In [2]:
# se crea un archivo .txt para almacenar los nombres de los archivos ya procesados
import os

if not os.path.exists('procesados.txt'):
    with open('procesados.txt', 'w') as file:
        pass # crea un archivo vacio

In [14]:
# lee los archivos procesados, por defecto nada
with open('procesados.txt', 'r') as file:
    procesados= set(file.read().splitlines())

procesados

{'223221647-ECN-BusinessPath-fulldoc.pdf'}

In [5]:
procesados_2= {}

In [ ]:
import os
import hashlib

ruta_docs_pdf= '../doc_pdf'
# carpeta_embeddings = ''

docs_no_procesados= []
# verificamos los archivos en carpeta de docs
for filename in os.listdir(ruta_docs_pdf):
    # verificar si el archivo se encuentra en procesados.txt
    if filename not in procesados:
        # print('El archivo no ha sido procesado')
        ruta_completa= os.path.join(ruta_docs_pdf,filename)
        docs_no_procesados.append(ruta_completa)
    else:
        print('Todos los archivos han sido procesados')

print(f'Documentos para procesar:\n  {docs_no_procesados}')
# luego que finalice todo el proceso, hay que realizar una función para agregar el archivo a procesados.txt
# with open('procesados.txt', 'w') as file:
#                 file.write(filename+"\n")

# ruta= os.path.join(ruta_docs_pdf, filename)

# doc_id2 = hashlib.md5(ruta.encode()).hexdigest() # codificamos la entrada string, aplicamos algoritmo y obtenemos salida hexadecimal


Todos los archivos han sido procesados
Documentos para procesar:
  ['../doc_pdf/601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf']


## Empieza el procesamiento

Extracción del texto 

In [5]:
from langchain_community.document_loaders import PyPDFLoader

def extraccion_page(ruta):
    loader = PyPDFLoader(ruta)
    pages = loader.load()
    # async for page in loader.alazy_load():
    #     pages.append(page)
    print('✅ Extracción realizada')
    return pages


Limpieza del texto

In [6]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'©.*?\n', '', text)  # remueve símbolos de copyright y similares
    text = re.sub(r'\n+', ' ', text)  # convierte múltiples saltos de línea en espacio
    text = re.sub(r'\s{2,}', ' ', text)  # remueve espacios extra
    return text.strip()


Creacción de la metadata, estructura planteada:
- documento_id
- nombre documento
- numero pagina
- total_pages

In [7]:
from pathlib import Path

def generate_metadata(ruta_completa, pages):
    filename = Path(ruta_completa).name
    document_id = hashlib.md5(filename.encode()).hexdigest() # codificamos solo el nombre del archivo
    total_pages = pages[0].metadata['total_pages']
    docs_metadata = []
    for page in pages:
        page_number = page.metadata['page_label'] # númeración correcta de la página
        
        metadata = {
            "document_id": document_id,
            "filename": filename,
            "page_number": page_number,
            "total_pages": total_pages,
        }
        page.page_content = clean_text(page.page_content) # cleaned_text = clean_text(page.page_content)
        
        docs_metadata.append(
            {
                'text': page.page_content, #cleaned_text, 
                'metadata': metadata
            }
        )
    print('✅ Generación Documentos con Metadata (Limpieza por página)')
    return docs_metadata

Generación de embeddings

In [8]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()
api_key=os.getenv('OPENAI_API_KEY')
# api_key
cliente= OpenAI()
cliente

In [ ]:
# from sentence_transformers import SentenceTransformer
import tiktoken # estimar la cantidad de token
import time

# modelo_seleccionado= SentenceTransformer('BAAI/bge-large-en-v1.5')
def costo_tokens(tokens):
    costo = tokens*0.02 /10**6 # 1 millon de tokens equivale a 0.02 dólares

    return f'   Tokens: {tokens}\n   Costo Tokens: ${costo:.4f}'


def generate_embedd(docs_metadata):
    modelo_openai = "text-embedding-3-small"
    encoding= tiktoken.encoding_for_model(modelo_openai)

    docs_embedd = []
    total_tokens= 0


    for doc in docs_metadata:
        texto= doc['text']

        # Generación de número de tokens
        tokens= encoding.encode(texto)
        nro_tokens = len(tokens)
        total_tokens += nro_tokens

        doc['metadata']['token']=nro_tokens # añado los tokens a la metadata
    
        start= time.time()
        #  Generación de embeddings
        response = cliente.embeddings.create(
            input= texto, 
            model = modelo_openai
        )
        finish= time.time()
        embedding= response.data[0].embedding
        
        # embedding= modelo_seleccionado.encode(doc['text'], normalize_embeddings= True)
        docs_embedd.append({
            'vector': embedding,  #con openai, directamente el embedding
            'text': texto, 
            'metadata': doc['metadata'] 

        })
    print(costo_tokens(total_tokens))
    print(f'Tiempo del embedding: {finish-start:.4f} segundos')
    print('✅ Generación de Embeddings')
    return docs_embedd

# comprobar con lo que sale en playground

Exportación embedding

In [10]:
ruta_completa

'../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf'

In [18]:
filename = Path(ruta_completa).name

import json
def exportacion_json(docs_embeding,filename):
    with open(f"../json_embedding/{filename}.json", "w", encoding="utf-8") as file:
        json.dump(docs_embeding, file, ensure_ascii=False, indent=2)

    # luego que finalice todo el proceso, hay que realizar una función para agregar el archivo a procesados.txt
    with open('procesados.txt', 'w') as file:
        file.write(filename+"\n")

    print('✅ Exportacción realizada')

# exportacion_json(filename)

Función completa 

In [ ]:
docs_no_procesados

In [ ]:
import tqdm
prueba = ['../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf']

# for ruta_archivo in  tqdm.tqdm(prueba):#docs_no_procesados:
def proceso_completo(docs_no_procesados):
    '''
    Parámetro de entrada: Lista con todas las rutas de los archivos no procesados
    '''
    for ruta_archivo in  prueba:#docs_no_procesados:
        filename = Path(ruta_completa).name
        # definir una funcion para aplicar  
        time1= time.time()
        pags=extraccion_page(ruta_archivo)
        docs_metadata = generate_metadata(ruta_archivo, pags)
        docs_embedd= generate_embedd(docs_metadata)
        exportacion_json(docs_embedd,filename)
        time3=time.time()
        print(f'   Tiempo total por {ruta_archivo}: {time3-time1:.2f} segundos')
        print('🎉 Realizado: Embeddings Generados Correctamente.\n')

# Funcion completa
# def procesamiento():
    # extracion texto
    # limpieza por pagina
    # creacion de docs_metadata
    # obtencion de embedding
    # exportación de embedding
    

✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
Tokens: 1938
Costo Tokens: $0.0000
Tiempo del embedding: 0.1533 segundos
✅ Generación de Embeddings
✅ Exportacción realizada
Tiempo total por ../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf: 1.45 segundos
🎉 Realizado: extracción, limpieza del texto, generación de metadata y embeddings.



In [108]:
len(docs_embedd[0]['vector'])

1536

Ya ahora que tengo el embedding demo vamos a modularizar

In [3]:
import langchain
import langchain_community